# Generative Adversarial Networks (GANs)

> Based on Stanford CS230 — LN7, Goodfellow et al. (2014)

A GAN pits two networks against each other: a **Generator** $G$ that synthesises fake data, and a **Discriminator** $D$ that tries to distinguish real from fake. At equilibrium, $G$ produces samples indistinguishable from the real data distribution.

## Learning Objectives

1. State the GAN minimax objective and explain the generator/discriminator roles
2. Understand training dynamics and common failure modes (mode collapse, vanishing gradients)
3. Implement and train a GAN on a 1-D toy distribution
4. Compare original GAN loss with the Wasserstein GAN (WGAN) objective

## GAN Objective

$$\min_G \max_D\; V(D, G) = \mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

- $D(x) \in (0,1)$ — probability that $x$ is real
- $z \sim \mathcal{N}(0, I)$ — latent noise vector
- Generator minimises $\log(1-D(G(z)))$ — or equivalently maximises $\log D(G(z))$ (non-saturating variant)

## Architecture

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 680 160" width="680" height="160" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <defs>
    <marker id="ga" markerWidth="7" markerHeight="7" refX="5" refY="3" orient="auto">
      <path d="M0,0 L0,6 L7,3 z" fill="#888"/>
    </marker>
  </defs>
  <!-- Noise -->
  <rect x="10" y="60" width="80" height="40" rx="4" fill="#56b6c2" fill-opacity="0.15" stroke="#56b6c2" stroke-width="1.5"/>
  <text x="50" y="84" text-anchor="middle" fill="#2a7080" font-size="10" font-weight="bold">z ~ N(0,I)</text>
  <!-- Generator -->
  <rect x="115" y="50" width="120" height="60" rx="5" fill="#7ecba1" fill-opacity="0.18" stroke="#7ecba1" stroke-width="1.6"/>
  <text x="175" y="77" text-anchor="middle" fill="#3d7a5e" font-size="11" font-weight="bold">Generator G</text>
  <text x="175" y="95" text-anchor="middle" fill="#3d7a5e" font-size="9">z → G(z) ≈ x_fake</text>
  <!-- Fake sample -->
  <rect x="265" y="60" width="85" height="40" rx="4" fill="#e05c5c" fill-opacity="0.12" stroke="#e05c5c" stroke-width="1.4"/>
  <text x="307" y="84" text-anchor="middle" fill="#b03a3a" font-size="10" font-weight="bold">G(z) fake</text>
  <!-- Real data -->
  <rect x="265" y="110" width="85" height="35" rx="4" fill="#5b9bd5" fill-opacity="0.15" stroke="#5b9bd5" stroke-width="1.4"/>
  <text x="307" y="131" text-anchor="middle" fill="#3a7bbf" font-size="10" font-weight="bold">x real</text>
  <!-- Discriminator -->
  <rect x="385" y="55" width="130" height="65" rx="5" fill="#c678dd" fill-opacity="0.15" stroke="#c678dd" stroke-width="1.6"/>
  <text x="450" y="82" text-anchor="middle" fill="#8a3ab8" font-size="11" font-weight="bold">Discriminator D</text>
  <text x="450" y="98" text-anchor="middle" fill="#8a3ab8" font-size="9">x → D(x) ∈ (0,1)</text>
  <text x="450" y="111" text-anchor="middle" fill="#8a3ab8" font-size="8">real=1 / fake=0</text>
  <!-- Output -->
  <rect x="545" y="65" width="110" height="45" rx="4" fill="#f4b942" fill-opacity="0.15" stroke="#f4b942" stroke-width="1.5"/>
  <text x="600" y="85" text-anchor="middle" fill="#a07010" font-size="10" font-weight="bold">Loss signals</text>
  <text x="600" y="100" text-anchor="middle" fill="#a07010" font-size="9">→ update D &amp; G</text>
  <!-- Arrows -->
  <line x1="90"  y1="80"  x2="115" y2="80"  stroke="#aaa" stroke-width="1.2" marker-end="url(#ga)"/>
  <line x1="235" y1="80"  x2="265" y2="80"  stroke="#aaa" stroke-width="1.2" marker-end="url(#ga)"/>
  <line x1="350" y1="80"  x2="385" y2="85"  stroke="#aaa" stroke-width="1.2" marker-end="url(#ga)"/>
  <line x1="350" y1="127" x2="385" y2="105" stroke="#aaa" stroke-width="1.2" marker-end="url(#ga)"/>
  <line x1="515" y1="88"  x2="545" y2="88"  stroke="#aaa" stroke-width="1.2" marker-end="url(#ga)"/>
</svg>

## Wasserstein GAN

WGAN replaces the JS divergence with Earth Mover (Wasserstein) distance, providing more stable gradients:

$$\min_G \max_{D \in \mathcal{F}} \mathbb{E}_{x \sim p_r}[D(x)] - \mathbb{E}_{z \sim p_z}[D(G(z))]$$

where $\mathcal{F}$ is the set of 1-Lipschitz functions (enforced via weight clipping or gradient penalty).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

sigmoid     = lambda z: 1 / (1 + np.exp(-np.clip(z, -30, 30)))
sigmoid_d   = lambda z: sigmoid(z) * (1 - sigmoid(z))

def relu(z): return np.maximum(0, z)
def relu_d(z): return (z > 0).astype(float)

# ---- Minimal 1-D GAN on a Gaussian mixture target ----
class MLP:
    """2-layer MLP for G or D."""
    def __init__(self, n_in, n_h, n_out, seed=0):
        np.random.seed(seed)
        s = 0.1
        self.W1 = np.random.randn(n_h, n_in) * s
        self.b1 = np.zeros(n_h)
        self.W2 = np.random.randn(n_out, n_h) * s
        self.b2 = np.zeros(n_out)

    def forward(self, x):
        self.z1 = self.W1 @ x + self.b1
        self.a1 = relu(self.z1)
        self.z2 = self.W2 @ self.a1 + self.b2
        return self.z2

    def forward_sigmoid(self, x):
        return sigmoid(self.forward(x))

def train_gan(n_iter=2000, lr=0.002, n_h=32, seed=42):
    np.random.seed(seed)
    G = MLP(1, n_h, 1, seed=0)
    D = MLP(1, n_h, 1, seed=1)

    d_losses, g_losses, snapshots = [], [], []

    for it in range(n_iter):
        # ---- Update D ----
        # Real samples: Gaussian mixture N(-2,0.5²) and N(2,0.5²)
        x_real = np.random.randn(1) * 0.5 + (2 if np.random.rand() > 0.5 else -2)
        z      = np.random.randn(1)
        x_fake = G.forward(z)

        d_real = D.forward_sigmoid(x_real)
        d_fake = D.forward_sigmoid(x_fake)

        # D loss: -log D(x_real) - log(1 - D(x_fake))
        d_loss = -np.log(d_real + 1e-8) - np.log(1 - d_fake + 1e-8)

        # Backprop D on real
        dL_dz2_real = -(1 - d_real)
        dL_da1_real = D.W2.T * dL_dz2_real
        dL_dW2_real = dL_dz2_real * D.a1
        dL_dW1_real = (dL_da1_real * relu_d(D.z1))[:, None] @ x_real[None, :]

        # Re-forward for fake (already done above)
        D.forward_sigmoid(x_fake)
        dL_dz2_fake = d_fake
        dL_dW2_fake = dL_dz2_fake * D.a1
        dL_da1_fake = D.W2.T * dL_dz2_fake
        dL_dW1_fake = (dL_da1_fake * relu_d(D.z1))[:, None] @ x_fake[None, :]

        D.W1 -= lr * (dL_dW1_real + dL_dW1_fake)
        D.W2 -= lr * (dL_dW2_real.reshape(D.W2.shape) + dL_dW2_fake.reshape(D.W2.shape))

        # ---- Update G (non-saturating: maximise log D(G(z))) ----
        z      = np.random.randn(1)
        x_fake = G.forward(z)
        d_fake_g = D.forward_sigmoid(x_fake)
        g_loss   = -np.log(d_fake_g + 1e-8)

        dL_dz2_d = d_fake_g                  # gradient through D's sigmoid
        dx_fake  = (D.W1.T @ (relu_d(D.z1) * (D.W2.T * dL_dz2_d).flatten()))
        # Backprop into G
        dL_dz2_g = -dx_fake
        dL_da1_g = G.W2.T * dL_dz2_g
        dL_dW2_g = dL_dz2_g * G.a1
        dL_dW1_g = (dL_da1_g * relu_d(G.z1))[:, None] @ z[None, :]

        G.W1 -= lr * dL_dW1_g
        G.W2 -= lr * dL_dW2_g.reshape(G.W2.shape)

        d_losses.append(float(d_loss))
        g_losses.append(float(g_loss))

        if it in [0, 100, 500, 999, 1999]:
            zs = np.random.randn(500, 1)
            fakes = np.array([G.forward(z)[0] for z in zs])
            snapshots.append((it, fakes.copy()))

    return G, D, d_losses, g_losses, snapshots

G, D, d_losses, g_losses, snapshots = train_gan()

# ---- Plots ----
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('1-D GAN Training on Gaussian Mixture', fontsize=13, fontweight='bold')

# Loss curves
ax = axes[0, 0]
window = 50
dl_smooth = np.convolve(d_losses, np.ones(window)/window, 'valid')
gl_smooth = np.convolve(g_losses, np.ones(window)/window, 'valid')
ax.plot(dl_smooth, color=P[1], lw=2, label='D loss')
ax.plot(gl_smooth, color=P[3], lw=2, label='G loss')
ax.set_xlabel('Iteration'); ax.set_ylabel('Loss'); ax.set_title('Training Losses')
ax.legend(); ax.grid(True)

# Generator output evolution
true_x = np.concatenate([np.random.randn(250)*0.5+2, np.random.randn(250)*0.5-2])
for i, (it, fakes) in enumerate(snapshots):
    ax = axes[i // 3 + (0 if i < 3 else 1), (i % 3) + (0 if i < 3 else 0)]
    if i == 0:
        ax = axes[0, 1]
    elif i == 1:
        ax = axes[0, 2]
    elif i == 2:
        ax = axes[1, 0]
    elif i == 3:
        ax = axes[1, 1]
    else:
        ax = axes[1, 2]
    ax.hist(true_x, bins=40, density=True, color=P[0], alpha=0.5, label='Real')
    ax.hist(fakes,  bins=40, density=True, color=P[1], alpha=0.5, label='Generated')
    ax.set_title(f'Iteration {it+1}')
    ax.set_xlabel('x'); ax.set_ylabel('density')
    ax.legend(fontsize=8); ax.grid(True)

plt.tight_layout()
plt.show()

# Discriminator boundary
x_range = np.linspace(-5, 5, 300).reshape(-1, 1)
d_scores = np.array([D.forward_sigmoid(x)[0] for x in x_range])
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x_range, d_scores, color=P[5], lw=2, label='D(x) — real probability')
ax.axhline(0.5, color='#bbb', ls='--', lw=1, label='D=0.5 (indistinguishable)')
ax.fill_between(x_range.flatten(), 0, d_scores, alpha=0.15, color=P[5])
ax.set_xlabel('x'); ax.set_ylabel('D(x)'); ax.set_title('Discriminator Score over Input Space (after training)')
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.show()
